In [ ]:
# Importa la librería pandas para manipulación de datos
import pandas as pd

In [ ]:
# Carga el archivo CSV 'sales_data_sample.csv' en un DataFrame de pandas
# La codificación 'latin1' se usa para manejar caracteres especiales si es necesario
df = pd.read_csv("/content/sample_data/sales_data_sample.csv", encoding="latin1")

In [ ]:
# Muestra las primeras 5 filas del DataFrame para una vista previa de los datos
df.head()

In [ ]:
# Muestra un resumen conciso del DataFrame, incluyendo tipos de datos, valores no nulos y uso de memoria
df.info()

In [ ]:
# Calcula y muestra el número de valores nulos (NaN) por columna
df.isnull().sum()

Primero cargamos el dataset usando pandas para analizar la información contenida en el archivo CSV. Luego verificamos columnas, tipos de datos y valores vacíos para preparar correctamente el corpus que usará el sistema RAG.

In [ ]:
# Instala las librerías necesarias para el procesamiento de lenguaje natural y RAG
# pandas: manipulación de datos
# langchain, langchain-community, langchain-experimental, langchain-ollama: componentes de LangChain para construir aplicaciones LLM
# tabulate: para formatear tablas
# mistralai: para interactuar con modelos de Mistral AI
!pip install pandas langchain langchain-community langchain-experimental tabulate langchain-ollama mistralai

In [ ]:
# Importa la librería pandas para manipulación de datos
import pandas as pd

# Carga el archivo CSV 'sales_data_sample.csv' en un DataFrame de pandas
# La codificación 'latin1' se usa para manejar caracteres especiales si es necesario
df = pd.read_csv(
    "/content/sample_data/sales_data_sample.csv",
    encoding="latin1"
)

In [ ]:
# Importa la librería pandas
import pandas as pd

# Define la ruta al archivo CSV
file_path = "/content/sample_data/sales_data_sample.csv"

# Carga el archivo CSV en un DataFrame de pandas
df = pd.read_csv(file_path, encoding="latin1")

# Muestra las primeras filas del DataFrame para verificar la carga
df.head()

In [ ]:
# Eliminar valores nulos importantes
df = df.fillna("No especificado")

# Convertir fechas correctamente
df["ORDERDATE"] = pd.to_datetime(df["ORDERDATE"])

# Verificar datos limpios
df.info()

In [ ]:
# Inicializa una lista vacía para almacenar los documentos (textos formateados)
documents = []

# Itera sobre cada fila del DataFrame para crear un string formateado para cada orden
for _, row in df.iterrows():
    # Formatea la información de la fila en un string legible
    text = f"""
    Orden: {row['ORDERNUMBER']}
    Cliente: {row['CUSTOMERNAME']}
    Ciudad: {row['CITY']}
    País: {row['COUNTRY']}
    Producto: {row['PRODUCTLINE']}
    Cantidad: {row['QUANTITYORDERED']}
    Total de venta: {row['SALES']} USD
    Estado: {row['STATUS']}
    Fecha: {row['ORDERDATE']}
    """

    # Agrega el string formateado a la lista de documentos, eliminando espacios en blanco al inicio/final
    documents.append(text.strip())

# Muestra la cantidad total de documentos creados
len(documents)

In [ ]:
# Importa el RecursiveCharacterTextSplitter de LangChain para dividir textos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Inicializa el "text splitter" con un tamaño de "chunk" (fragmento) de 300 caracteres
# y una superposición de 50 caracteres entre "chunks" adyacentes
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

# Divide los documentos en "chunks" (fragmentos de texto) utilizando el "text splitter"
chunks = text_splitter.create_documents(documents)

# Muestra la cantidad total de "chunks" creados
len(chunks)

In [ ]:
# Instala las librerías `faiss-cpu` y `sentence-transformers`.
# `faiss-cpu` es una biblioteca para búsqueda de similitud eficiente y agrupamiento de vectores densos.
# `sentence-transformers` proporciona modelos pre-entrenados para generar embeddings de oraciones.
!pip install faiss-cpu sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Inicializa el modelo de embeddings de Hugging Face.
# Se utiliza 'sentence-transformers/all-MiniLM-L6-v2' que es un modelo eficiente para generar embeddings semánticos de texto.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
from langchain_community.vectorstores import FAISS

# Crea un almacén de vectores FAISS a partir de los 'chunks' de documentos y los embeddings generados.
# FAISS (Facebook AI Similarity Search) es una librería para una búsqueda eficiente de similitud en grandes colecciones de vectores.
vectorstore = FAISS.from_documents(chunks, embeddings)

In [ ]:
# Define la consulta que se utilizará para buscar documentos similares en el almacén de vectores.
query = "ventas en Nueva York"

# Realiza una búsqueda de similitud en el 'vectorstore' usando la 'query'.
# Se solicitan los 3 documentos más similares (k=3).
docs = vectorstore.similarity_search(query, k=3)

# Itera sobre los documentos encontrados y muestra su contenido.
# Cada documento representa una "Orden" de venta relevante para la consulta.
for d in docs:
    print(d.page_content)
    print("-" * 50)

In [ ]:
import requests
from google.colab import userdata
userdata.get('MISTRAL_API_KEY')

MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')

def ask_mistral(prompt):
    url = "https://api.mistral.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "mistral-small-latest",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"]

In [ ]:
# Librería para hacer peticiones HTTP
import requests
from google.colab import userdata
userdata.get('MISTRAL_API_KEY')

# 🔑 API KEY de Mistral (NO compartir públicamente)
MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')

# URL del endpoint oficial de chat completions
MISTRAL_URL = "https://api.mistral.ai/v1/chat/completions"

In [ ]:
def call_mistral(prompt):
    """
    Envía un prompt a Mistral y devuelve la respuesta del modelo.
    """

    # Headers de autenticación
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }

    # Datos que enviamos al modelo
    data = {
        "model": "mistral-small-latest",  # modelo rápido y barato
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    # Petición POST a la API
    response = requests.post(MISTRAL_URL, headers=headers, json=data)

    # Convertimos respuesta a JSON
    result = response.json()

    # Devolvemos el texto generado
    return result["choices"][0]["message"]["content"]

In [ ]:
# Prueba simple para verificar conexión
test_prompt = "Explícame qué es un RAG en inteligencia artificial en 3 líneas."

print(call_mistral(test_prompt))

In [ ]:
# ==========================================
# FUNCIÓN PRINCIPAL DEL BOT RAG
# ==========================================

def ask_bot(question):
    """
    Recibe una pregunta del usuario,
    busca información relevante en FAISS
    y usa Mistral para generar la respuesta.
    """

    # --------------------------------------
    # 1. Buscar contexto relevante en FAISS
    # --------------------------------------
    docs = vectorstore.similarity_search(question, k=3)

    # --------------------------------------
    # 2. Unir documentos encontrados
    # --------------------------------------
    context = "\n\n".join([doc.page_content for doc in docs])

    # --------------------------------------
    # 3. Crear prompt para Mistral
    # --------------------------------------
    prompt = f"""
    Eres un asistente experto en análisis de ventas.

    Usa únicamente la información del CONTEXTO.

    CONTEXTO:
    {context}

    PREGUNTA:
    {question}

    Responde de forma clara y breve.
    """

    # --------------------------------------
    # 4. Enviar prompt a Mistral
    # --------------------------------------
    answer = call_mistral(prompt)

    # --------------------------------------
    # 5. Retornar respuesta final
    # --------------------------------------
    return answer

In [ ]:
# Pregunta de prueba
question = "¿Qué información hay sobre ventas en España?"

# Ejecutar bot
response = ask_bot(question)

# Mostrar respuesta
print(response)

In [ ]:
# ==========================================
# CHAT INTERACTIVO
# ==========================================

while True:

    # Pedir pregunta al usuario
    question = input("Tú: ")

    # Comando para salir
    if question.lower() == "salir":
        print("Chat finalizado.")
        break

    # Obtener respuesta del bot
    response = ask_bot(question)

    # Mostrar respuesta
    print("\nBot:", response)
    print("-" * 50)